# Pratica — Lezione 11: sistema il "traduttore" di Babel Fish

Nella lezione hai visto che il progetto reale di Babel Fish (`babel-fish-project/translator-with-voice-and-watsonx/worker.py`) ha un bug reale: la funzione `watsonx_process_message`, che dovrebbe tradurre il messaggio dell'utente, in realtà usa un prompt generico che non garantisce nessuna traduzione:

```python
def watsonx_process_message(user_message):
    prompt = f"""Respond to the query: ```{user_message}```"""
    response_text = model.generate_text(prompt=prompt)
    return response_text.strip()
```

In questa pratica risolverai lo stesso problema, ma con la via 100% gratuita che hai usato in lezione: invece di affidarti a un prompt ben scritto per un LLM generalista, costruirai una funzione che riceve un testo in inglese e una lingua di destinazione, e che **garantisce** la traduzione usando la pipeline di traduzione di Hugging Face.

## Esercizio — Costruisci un traduttore che traduca davvero

**Contesto reale**: il bug di Babel Fish esiste perché nessuno ha verificato che il prompt `"Respond to the query: ..."` producesse una traduzione reale per input diversi. Il tuo compito è costruire un'alternativa che non dipenda dalla buona volontà di un prompt.

**Suggerimenti**:
- Hai già la pipeline `translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")` della lezione, che traduce inglese → spagnolo.
- Per tradurre verso **altre** lingue di destinazione, Hugging Face ha modelli equivalenti con lo stesso pattern di nome: `Helsinki-NLP/opus-mt-en-fr` (francese), `Helsinki-NLP/opus-mt-en-de` (tedesco), `Helsinki-NLP/opus-mt-en-it` (italiano), ecc. Il pattern generale è `Helsinki-NLP/opus-mt-en-{codigo_idioma}`.
- La tua funzione deve ricevere il testo in inglese e la lingua di destinazione (ad esempio, `"es"`, `"fr"`, `"de"`), e restituire il testo tradotto, caricando il modello corrispondente a quella lingua.
- Per non ricaricare lo stesso modello più volte, puoi salvare le pipeline già caricate in un dizionario (`cache`).

Completa la funzione `traducir_a_idioma` qui sotto — non riutilizzare la versione "bug" del prompt generico.

In [ ]:
from transformers import pipeline

# Cache delle pipeline gia caricate, per non scaricare lo stesso modello due volte.
_pipelines_cache = {}

def traducir_a_idioma(texto_ingles, idioma_destino):
    """
    Traduce un testo in inglese nella lingua di destinazione indicata, usando un
    modello di traduzione specializzato di Hugging Face (mai un prompt
    generico che "potrebbe" tradurre o no).

    texto_ingles: il testo di input, in inglese.
    idioma_destino: codice della lingua di destinazione, ad esempio "es", "fr", "de".

    Deve restituire il testo gia tradotto (str).
    """
    # TODO 1: costruisci il nome del modello di Hugging Face per questa lingua
    #         di destinazione, seguendo il pattern "Helsinki-NLP/opus-mt-en-{idioma_destino}"
    nombre_modelo = None  # <-- completa qui

    # TODO 2: se la pipeline per questa lingua e gia in _pipelines_cache, riutilizzala;
    #         altrimenti creala con pipeline(f"translation_en_to_{idioma_destino}", model=nombre_modelo)
    #         e salvala in _pipelines_cache prima di usarla.


    # TODO 3: usa la pipeline per tradurre texto_ingles e restituisci
    #         solo il testo tradotto (stringa), non la lista/dizionario completo.
    return None  # <-- sostituisci questo con il risultato reale


# Prova la tua funzione con almeno due lingue di destinazione diverse:
# print(traducir_a_idioma("Where is the nearest train station?", "es"))
# print(traducir_a_idioma("Where is the nearest train station?", "fr"))

<details>
<summary>💡 Vedi soluzione</summary>

```python
from transformers import pipeline

_pipelines_cache = {}

def traducir_a_idioma(texto_ingles, idioma_destino):
    """
    Traduce un testo in inglese nella lingua di destinazione indicata, usando un
    modello di traduzione specializzato di Hugging Face.
    """
    # Costruiamo il nome del modello seguendo il pattern di Helsinki-NLP
    nombre_modelo = f"Helsinki-NLP/opus-mt-en-{idioma_destino}"

    # Riutilizziamo la pipeline se l'abbiamo gia caricata prima, altrimenti la creiamo
    if idioma_destino not in _pipelines_cache:
        _pipelines_cache[idioma_destino] = pipeline(
            f"translation_en_to_{idioma_destino}",
            model=nombre_modelo,
        )
    translator = _pipelines_cache[idioma_destino]

    # Eseguiamo la traduzione e restituiamo solo il testo
    resultado = translator(texto_ingles)
    return resultado[0]["translation_text"]


print(traducir_a_idioma("Where is the nearest train station?", "es"))
print(traducir_a_idioma("Where is the nearest train station?", "fr"))
```

A differenza del prompt generico di Babel Fish (`"Respond to the query: ..."`), questa funzione non può "dimenticarsi" di tradurre: il modello che carica è addestrato specificamente per il compito di traduzione dall'inglese verso una lingua concreta, quindi la traduzione è garantita dal design stesso del modello, non dalla fortuna di un prompt scritto bene (o male).
</details>

## Per continuare

- Notebook di lezione: [clase11_traduccion_babelfish.ipynb](../notebooks_clase/clase11_traduccion_babelfish_IT.ipynb)
- Capitolo del manuale: [15_traduccion_babelfish.md](../manual/15_traduccion_babelfish_IT.md)